# Part A — EDA and preprocessing

In [15]:
import pandas as pd

In [16]:
credit_risk_df = pd.read_csv("credit_applicants.csv")

In [17]:
credit_risk_df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   applicant_id              400 non-null    object 
 1   age                       400 non-null    int64  
 2   monthly_income_inr        400 non-null    int64  
 3   existing_loans_count      400 non-null    int64  
 4   credit_utilization_ratio  400 non-null    float64
 5   upi_monthly_inflow_inr    400 non-null    int64  
 6   bounced_payments_count    400 non-null    int64  
 7   credit_bureau_score       320 non-null    float64
 8   employment_type           400 non-null    object 
 9   default                   400 non-null    int64  
dtypes: float64(2), int64(6), object(2)
memory usage: 31.4+ KB


In [18]:
#report the exact measured rate
credit_risk_df['default'].value_counts(normalize=True)

default
0    0.7975
1    0.2025
Name: proportion, dtype: float64

In [19]:
# and the exact percentage of missing credit_bureau_score values
credit_risk_df['credit_bureau_score'].isna().mean() * 100 

20.0

In [20]:
#a binary is_thin_file flag (1 where credit_bureau_score is missing, 0 otherwise)
# ...existing code...
credit_risk_df['is_thin_file'] = credit_risk_df['credit_bureau_score'].isna().astype(int)

In [21]:
credit_risk_df['is_thin_file'].value_counts(normalize=True)

is_thin_file
0    0.8
1    0.2
Name: proportion, dtype: float64

# Part B — Classification models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features and target
X = credit_risk_df.drop(columns=['default', 'applicant_id'])
y = credit_risk_df['default']

# Stratified Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    stratify=y, 
    random_state=42
)

# Median Imputation (Fit on Train, Apply to Both)
train_bureau_median = X_train['credit_bureau_score'].median()

X_train['credit_bureau_score'] = X_train['credit_bureau_score'].fillna(train_bureau_median)
X_test['credit_bureau_score'] = X_test['credit_bureau_score'].fillna(train_bureau_median)

# Encode Categorical Feature (One-Hot Encoding)
X_train = pd.get_dummies(X_train, columns=['employment_type'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['employment_type'], drop_first=True)

# Scale Numeric Features
numeric_cols = [
    'age', 'monthly_income_inr', 'existing_loans_count', 
    'credit_utilization_ratio', 'upi_monthly_inflow_inr', 
    'bounced_payments_count', 'credit_bureau_score'
]

scaler = StandardScaler()
# Fit AND transform on training data
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
# ONLY transform on test data
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# 1. Initialize the classifiers
log_reg = LogisticRegression(random_state=42)
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100)

# 2. Train both models on the identical training split
log_reg.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)

# 3. Generate predictions and probability scores for the test set
lr_preds = log_reg.predict(X_test)
lr_probs = log_reg.predict_proba(X_test)[:, 1]

rf_preds = rf_clf.predict(X_test)
rf_probs = rf_clf.predict_proba(X_test)[:, 1]

# 4. Compile the evaluation suite side-by-side
evaluation_suite = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Logistic Regression': [
        accuracy_score(y_test, lr_preds),
        precision_score(y_test, lr_preds, zero_division=0),
        recall_score(y_test, lr_preds),
        f1_score(y_test, lr_preds),
        roc_auc_score(y_test, lr_probs)
    ],
    'Random Forest': [
        accuracy_score(y_test, rf_preds),
        precision_score(y_test, rf_preds, zero_division=0),
        recall_score(y_test, rf_preds),
        f1_score(y_test, rf_preds),
        roc_auc_score(y_test, rf_probs)
    ]
})

# Format the numbers for clean reading in your notebook
for col in ['Logistic Regression', 'Random Forest']:
    evaluation_suite[col] = evaluation_suite[col].round(4)

print(evaluation_suite.to_string(index=False))

   Metric  Logistic Regression  Random Forest
 Accuracy               0.7600         0.7600
Precision               0.3889         0.3000
   Recall               0.3500         0.1500
 F1-Score               0.3684         0.2000
  ROC-AUC               0.7188         0.6388


# Risk-based pricing table:

In [26]:
# Combine the actual test results and the model's predicted probabilities
risk_df = pd.DataFrame({
    'Actual_Default': y_test,
    'Predicted_Probability': rf_probs # Using the Random Forest probabilities
})

# Divide the applicants into 4 equal-sized tiers based on probability
risk_df['Risk_Tier'] = pd.qcut(
    risk_df['Predicted_Probability'], 
    q=4, 
    labels=['Tier 1 (Lowest Risk)', 'Tier 2 (Medium Risk)', 'Tier 3 (High Risk)', 'Tier 4 (Highest Risk)']
)

# Calculate the actual observed default rate for each tier
pricing_table = risk_df.groupby('Risk_Tier', observed=False).agg(
    Applicant_Count=('Actual_Default', 'size'),
    Observed_Default_Rate=('Actual_Default', 'mean') # Mean of 1s and 0s equals the rate
).reset_index()

# Format the output for the dashboard/report
pricing_table['Observed_Default_Rate'] = (pricing_table['Observed_Default_Rate'] * 100).map("{:.1f}%".format)

print(pricing_table.to_string(index=False))

            Risk_Tier  Applicant_Count Observed_Default_Rate
 Tier 1 (Lowest Risk)               28                  7.1%
 Tier 2 (Medium Risk)               24                 16.7%
   Tier 3 (High Risk)               23                 30.4%
Tier 4 (Highest Risk)               25                 28.0%


# Part C — Anomaly detection and optional segmentation



In [27]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. Load data and calculate contamination
txn_df = pd.read_csv("txn_behaviour.csv")
total_rows = len(txn_df)
seeded_anomalies_count = 15

# Contamination = 15 / 265 = 0.0566
contamination_rate = seeded_anomalies_count / total_rows

# 2. Preprocess features
# Keep IDs aside for evaluation later
txn_ids = txn_df['txn_id']

# Drop IDs from the training features
X_txn = txn_df.drop(columns=['txn_id', 'applicant_id'])

# Encode 'channel' (P2P vs P2M)
X_txn = pd.get_dummies(X_txn, columns=['channel'], drop_first=True)

# Standardize the features
scaler_iso = StandardScaler()
X_txn_scaled = scaler_iso.fit_transform(X_txn)

# 3. Initialize and fit Isolation Forest
iso_forest = IsolationForest(contamination=contamination_rate, random_state=42)
txn_df['anomaly_prediction'] = iso_forest.fit_predict(X_txn_scaled)

# 4. Calculate Recall explicitly on the seeded anomalies
# Isolate the 15 specific rows we know are anomalies
seeded_anomalies = txn_df[txn_df['txn_id'].str.startswith('BTXNA')]

# Count how many the model successfully flagged (-1)
caught_count = (seeded_anomalies['anomaly_prediction'] == -1).sum()
recall_score = caught_count / seeded_anomalies_count

print(f"Exact Contamination Rate: {contamination_rate:.4f}")
print(f"Seeded Anomalies Caught: {caught_count} / {seeded_anomalies_count}")
print(f"Anomaly Detection Recall: {recall_score * 100:.1f}%")

Exact Contamination Rate: 0.0566
Seeded Anomalies Caught: 5 / 15
Anomaly Detection Recall: 33.3%


The Final Comparison and Recommendation :-
    recommend the Logistic Regression model. It beat the Random Forest across every meaningful metric.

The Risk-Based Pricing Table Check :- 
    Your tiers go from 7.1% -> 16.7% -> 30.4% -> 28.0%.

The Anomaly Detection Recall :- 
    You successfully caught 5 out of the 15 seeded anomalies, giving you a 33.3% Recall